# MJLab policy visualization + velocity probe (Colab)

Renders a trained MJLab checkpoint to video inline and runs a **command-tracking
probe** (commanded vs achieved base velocity). Run `mjlab_training_colab.ipynb` in
the same session first, or upload/mount your own checkpoint.

Why probe: on curriculum locomotion tasks episode reward averages over the command
distribution and can stay high while the policy quietly stops executing parts of the
range (or never learns to walk at all — under-diversified runs converge to
reward-farming standers that score well and probe ~0.0 m/s). Trust the probe.

In [ ]:
# rl_games from git until the 2.0 PyPI release (then: %pip install rl-games mjlab)
RL_GAMES_REF = 'VM/feature/b1-ppo'   # flip to 'master' after the branch merges, PyPI after release

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    f'git+https://github.com/Denys88/rl_games.git@{RL_GAMES_REF}',
    'mjlab==1.5.0', 'warp-lang==1.14.0', 'mujoco-warp==3.10.0.1',
    'imageio[ffmpeg]', 'tensorboard'])
# warp pins: warp-lang 1.15 / mujoco-warp 3.10.0.2 currently crash mjlab's reset
# (CUDA illegal memory access in the warp<->torch mask interop) -- verified 2026-07-18.
# Drop the pins once fixed upstream.

import torch
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu} ({vram:.0f} GiB)")
if 'T4' in gpu:
    print("WARNING: T4 is untested for MuJoCo Warp and lacks bf16 support -- expect issues and "
          "slow training. Use an L4 or A100 runtime (Colab Pro) for the intended experience.")
# (rendering + probe need only a few envs -- any GPU tier works here)

In [ ]:
CHECKPOINT = 'runs/MJLab_Go1_colab/nn/MJLab_Go1_colab.pth'  # <- from the training notebook
TASK = 'Mjlab-Velocity-Flat-Unitree-Go1'
NUM_ENVS, STEPS = 3, 240

import glob
hits = sorted(glob.glob('runs/MJLab_Go1_colab*/nn/MJLab_Go1_colab.pth'))
if hits: CHECKPOINT = hits[-1]
print('using', CHECKPOINT)

In [ ]:
# Fetch the shipped Go1 config from the repo (the pip package ships code, not configs)
import urllib.request
CFG_URL = (f'https://raw.githubusercontent.com/Denys88/rl_games/{RL_GAMES_REF}'
           '/rl_games/configs/mjlab/ppo_go1_velocity.yaml')
urllib.request.urlretrieve(CFG_URL, 'ppo_go1_velocity.yaml')
CONFIG = 'ppo_go1_velocity.yaml'
print('fetched', CONFIG)

In [ ]:
import warp as wp
wp.init()
import numpy as np, torch, yaml
from mjlab.tasks.registry import load_env_cfg
from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv

cfg = load_env_cfg(TASK)
cfg.scene.num_envs = NUM_ENVS
cfg.viewer.width, cfg.viewer.height = 960, 540
cfg.viewer.distance, cfg.viewer.elevation = 6.0, -20.0
env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
obs, _ = env.reset()
key = 'actor' if 'actor' in obs else 'policy'

In [ ]:
from gymnasium import spaces
from rl_games.torch_runner import Runner

with open(CONFIG) as f:
    params = yaml.safe_load(f)['params']
params['config']['env_info'] = {
    'observation_space': spaces.Box(-np.inf, np.inf, (obs[key].shape[-1],), np.float32),
    'action_space': spaces.Box(-1.0, 1.0, (env.action_space.shape[-1],), np.float32),
    'agents': 1,
}
runner = Runner(); runner.load({'params': params})
player = runner.create_player()
player.restore(CHECKPOINT)
player.has_batch_dimension = True

In [ ]:
import imageio
frames = []
for t in range(STEPS):
    with torch.no_grad():
        action = player.get_action(obs[key], is_deterministic=True)
    obs, rew, term, trunc, info = env.step(action)
    frames.append(env.render())
imageio.mimwrite('mjlab_rollout.mp4', frames, fps=30, quality=8)

from IPython.display import Video
Video('mjlab_rollout.mp4', embed=True, width=720)

In [ ]:
# Velocity probe: healthy notebook-scale Go1 achieves ~0.6-0.9 at commanded 1.0;
# a reward-farming stander achieves ~0.0 (full-scale training reaches 0.9+).
COMMANDED_VX = 1.0
cmd = env.unwrapped.command_manager.get_term('twist')
obs, _ = env.reset()
vels = []
for t in range(STEPS):
    cmd.vel_command_b[:, 0] = COMMANDED_VX
    cmd.vel_command_b[:, 1:] = 0.0
    cmd.is_heading_env[:] = False; cmd.is_standing_env[:] = False
    cmd.is_world_env[:] = False; cmd.is_forward_env[:] = False
    with torch.no_grad():
        action = player.get_action(obs[key], is_deterministic=True)
    obs, rew, term, trunc, info = env.step(action)
    if t >= STEPS // 2:
        vels.append(cmd.robot.data.root_link_lin_vel_b[:, 0].clone())
v = torch.stack(vels)
print(f"commanded vx={COMMANDED_VX:.2f} -> achieved {v.mean().item():.3f} "
      f"+/- {v.std().item():.3f} m/s")
env.close()